In [75]:
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.utils import shuffle
from implicit import als
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle

Load Data

In [76]:
df = pd.read_csv(r"C:\Users\bhaga\.cache\kagglehub\datasets\grouplens\movielens-20m-dataset\versions\1\df.csv")

In [77]:
df = df.iloc[:,1:]

### ALS Model

Sample Users

In [78]:
users = np.random.choice(df.userId.unique(), size=10000, replace=False)

df_sample = df[df.userId.isin(users)]

df_sample["rating_adj"] = (df_sample["rating"] - df_sample.groupby("userId")["rating"].transform("mean"))

In [103]:
df["rating_adj"] = (df["rating"] - df.groupby("userId")["rating"].transform("mean"))

Take Above Average User Ratings

In [79]:
implicit_df = df_sample[df_sample["rating_adj"] >= 0.5].copy()

Train Test Split

In [80]:
train = []
test = []

for uid, g in implicit_df.groupby("userId"):
    if len(g) < 10:
        train.append(g)
        continue
    holdout = g.sample(n=2, random_state=42)
    test.append(holdout)
    train.append(g.drop(holdout.index))

train_df = pd.concat(train)
test_df = pd.concat(test)

In [117]:
# Encode userId and movieId to contiguous integer indices
user_cat = df['userId'].astype('category')
movie_cat = df['movieId'].astype('category')

user_idx = user_cat.cat.codes
movie_idx = movie_cat.cat.codes

user_map = dict(enumerate(user_cat.cat.categories))   # idx -> userId
movie_map = dict(enumerate(movie_cat.cat.categories)) # idx -> movieId

In [122]:
movie_cat

0               2
1              29
2              32
3              47
4              50
            ...  
20000258    68954
20000259    69526
20000260    69644
20000261    70286
20000262    71619
Name: movieId, Length: 20000263, dtype: category
Categories (26744, int64): [1, 2, 3, 4, ..., 131256, 131258, 131260, 131262]

In [105]:
alpha = 20

confidence = (1+alpha*df["rating_adj"].clip(lower=0))
df_matrix = csr_matrix((confidence,(user_idx,movie_idx)))

Train ALS Model

In [106]:
model = als.AlternatingLeastSquares(factors=50, regularization=1, iterations=20, use_gpu=False)
model.fit(df_matrix.T)

c:\Users\bhaga\anaconda3\envs\movie_rec\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.24808955192565918 seconds
  warnings.warn(
100%|██████████| 20/20 [00:33<00:00,  1.68s/it]


In [107]:
with open("als_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [108]:
user_factors = model.user_factors  # shape: (n_users, factors)
item_factors = model.item_factors  # shape: (n_items, factors)

print(f"User factors: {user_factors.shape}")
print(f"Item factors: {item_factors.shape}")


User factors: (26744, 50)
Item factors: (138493, 50)


In [109]:
movie_meta = (
    df
    .groupby("movieId")
    .agg({
        "vote_average": "mean",
        "popularity": "mean"
    })
)

ALS Recommend Model

In [110]:
movie_lookup = (df.drop_duplicates("movieId").set_index("movieId"))

movie_pop = (df.groupby("movieId").size())

def recommend(user_id, n=10):
    if user_id not in user_cat.cat.categories:
        raise ValueError("User not found")

    u_idx = user_cat.cat.categories.get_loc(user_id)

    ids, als_scores = model.recommend(
        userid=u_idx,
        user_items=df_matrix[u_idx],
        N=300,
        filter_already_liked_items=True
    )

    movie_ids = [movie_cat.cat.categories[i] for i in ids]

    recs = pd.DataFrame({
        "movieId": movie_ids,
        "als_score": als_scores
    })

    recs["num_ratings"] = (recs["movieId"].map(movie_pop).fillna(1))
    recs["pop_score"] = (np.log1p(recs["num_ratings"]))

    recs = recs.merge(
        movie_meta,
        on="movieId",
        how="left"
    )

    recs["vote_average"] = (recs["vote_average"].fillna(train_df["vote_average"].median()))
    recs["popularity"] = (recs["popularity"].fillna(0))
    recs["vote_norm"] = (recs["vote_average"] / recs["vote_average"].max())
    recs["tmdb_pop_norm"] = (recs["popularity"] / max(recs["popularity"].max(), 1))
    recs["als_norm"] = (recs["als_score"] / recs["als_score"].max())
    recs["pop_norm"] = (recs["pop_score"] / recs["pop_score"].max())


    recs["final_score"] = (
        0.65 * recs["als_norm"]
        + 0.20 * recs["pop_norm"]
        + 0.10 * recs["vote_norm"]
        + 0.05 * recs["tmdb_pop_norm"]
    )

    recs["title"] = (recs["movieId"].map(movie_lookup["title"]))
    recs = (recs.sort_values("final_score",ascending=False).head(n))

    return recs[["movieId","title","final_score","als_score","num_ratings","vote_average"]]

In [111]:
df[df['userId']==15][['title','rating']]

,title,rating
1805,GoldenEye (1995),2.0
1806,"American President, The (1995)",3.0
1807,Nixon (1995),2.0
1808,Sense and Sensibility (1995),3.0
1809,Get Shorty (1995),3.0
1810,Babe (1995),4.0
1811,Dead Man Walking (1995),3.0
1812,Clueless (1995),3.0
1813,"Usual Suspects, The (1995)",3.0
1814,"Birdcage, The (1996)",3.0


In [112]:
recommend(15)

IndexError: index 83089 is out of bounds for axis 0 with size 26744

### Content Recommendation Model

In [113]:
movies = df.drop_duplicates("movieId").copy()

cols = ["overview","genres","tags","genome_tags"]
for c in cols:
    movies[c] = (movies[c].fillna("").astype(str))

movies["content"] = (movies["genres"] + " " + movies["tags"] + " " + movies["genome_tags"] + " " + movies["overview"])

In [91]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

movie_embeddings = (
    embedder.encode(
        movies["content"].tolist(),
        show_progress_bar=True
    )
)

Batches: 100%|██████████| 836/836 [06:24<00:00,  2.18it/s]


In [92]:
import scipy

In [114]:
movie_to_idx = {m:i for i,m in enumerate(movies["movieId"])}

In [115]:
np.save("movie_embeddings.npy", movie_embeddings)
pickle.dump(movie_to_idx, open("movie_to_idx.pkl", "wb"))
pickle.dump(user_cat, open("user_cat.pkl", "wb"))
pickle.dump(movie_cat, open("movie_cat.pkl", "wb"))
pickle.dump(movie_pop, open("movie_pop.pkl", "wb"))
pickle.dump(movie_lookup, open("movie_lookup.pkl", "wb"))
train_matrix_sparse = scipy.sparse.save_npz("train_matrix.npz", df_matrix)
pickle.dump(movie_meta, open("movie_meta.pkl", "wb"))

In [96]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    movie_embeddings
)

In [97]:
def content_recommend(movie_id, n=10):
    idx = movie_to_idx[movie_id]
    sims = similarity[idx]
    best = (np.argsort(sims)[::-1][1:n+1])
    return movies.iloc[best][["movieId","title"]]

In [61]:
def content_user_vector(user_id):
    liked = train_df[(train_df.userId == user_id) & (train_df.rating >= 4)]
    liked = liked[liked.movieId.isin(movie_to_idx)]
    idxs = [movie_to_idx[m] for m in liked.movieId]
    weights = liked.set_index("movieId").loc[liked.movieId, "rating"].values
    return np.average(movie_embeddings[idxs], weights=weights, axis=0)

In [62]:
def recommend_merged(user_id, n=10):

    # ---------- validate ----------
    if user_id not in user_cat.cat.categories:
        raise ValueError("User not found")

    u_idx = user_cat.cat.categories.get_loc(user_id)

    # ---------- ALS candidate generation ----------
    ids, als_scores = model.recommend(
        userid=u_idx,
        user_items=train_matrix[u_idx],
        N=300,
        filter_already_liked_items=True
    )

    movie_ids = [movie_cat.cat.categories[i] for i in ids]

    recs = pd.DataFrame({
        "movieId": movie_ids,
        "als_score": als_scores
    })

    # ---------- content score ----------
    try:
        user_content = content_user_vector(user_id)
        def content_score(movie_id):
            if movie_id not in movie_to_idx:
                return 0
            idx = movie_to_idx[movie_id]
            return float(movie_embeddings[idx] @ user_content)
        recs["content_score"] = (recs["movieId"].apply(content_score))
    except:
        recs["content_score"] = 0

    # ---------- popularity ----------
    recs["num_ratings"] = (recs["movieId"].map(movie_pop).fillna(1))

    recs["pop_score"] = (np.log1p(recs["num_ratings"]))
    recs = recs[recs["num_ratings"] >= 50]

    # ---------- metadata ----------
    recs = recs.merge(movie_meta,on="movieId",how="left")
    recs["vote_average"] = (recs["vote_average"].fillna(train_df["vote_average"].median()))
    recs["popularity"] = (recs["popularity"].fillna(0))

    # ---------- normalization ----------
    recs["als_norm"] = (recs["als_score"] / max(recs["als_score"].max(),1e-8))

    if recs["content_score"].max() > 0:
        recs["content_norm"] = (recs["content_score"] / recs["content_score"].max())
    else:
        recs["content_norm"] = 0

    recs["pop_norm"] = (recs["pop_score"] / max(recs["pop_score"].max(), 1e-8))
    recs["vote_norm"] = (recs["vote_average"] / max(recs["vote_average"].max(),1))
    recs["tmdb_pop_norm"] = (recs["popularity"] / max(recs["popularity"].max(),1))

    # ---------- final hybrid score ----------
    recs["final_score"] = (0.40 * recs["als_norm"] + 0.40 * recs["content_norm"] + 0.10 * recs["pop_norm"] + 0.07 * recs["vote_norm"] + 0.03 * recs["tmdb_pop_norm"])

    # ---------- titles ----------
    recs["title"] = (recs["movieId"].map(movie_lookup["title"]))
    recs = (recs.sort_values("final_score", ascending=False).head(n))

    return recs[["movieId","title","final_score","als_score","content_score","num_ratings","vote_average"]]

In [34]:
train_df[train_df['userId']==15][['title','rating']]

,title,rating
1810,Babe (1995),4.0
1815,Apollo 13 (1995),4.0
1820,Hoop Dreams (1994),4.0
1828,Forrest Gump (1994),5.0
1829,"Lion King, The (1994)",4.0
1834,"Fugitive, The (1993)",4.0
1835,In the Line of Fire (1993),4.0
1840,"Remains of the Day, The (1993)",4.0
1841,Schindler's List (1993),5.0
1849,Snow White and the Seven Dwarfs (1937),4.0


In [63]:
recommend_merged(15)

,movieId,title,final_score,als_score,content_score,num_ratings,vote_average
0,3993,Quills (2000),0.778703,0.796776,0.286405,61,7.135
5,4326,Mississippi Burning (1988),0.762452,0.467755,0.437490,113,7.700
3,678,Some Folks Call It a Sling Blade (1993),0.747719,0.490536,0.426855,94,6.691
10,8783,"Village, The (2004)",0.720900,0.395940,0.448362,83,6.487
14,8957,Saw (2004),0.716552,0.373274,0.423880,157,7.400
36,110,Braveheart (1995),0.705161,0.252253,0.440173,1837,7.900
1,2770,Bowfinger (1999),0.698249,0.517100,0.354419,109,6.187
15,1179,"Grifters, The (1990)",0.697709,0.341956,0.444998,162,6.512
7,532,Serial Mom (1994),0.684599,0.417001,0.395287,83,6.730
4,1876,Deep Impact (1998),0.681874,0.470577,0.358039,97,6.238
